# 03: Assess Exposure

You found a CVE on one host. Now find out how many other hosts in the fleet have it open.

This tells you the blast radius: if the attacker used this vulnerability once, every other host
with the same package is exposed to the same attack.

In [ ]:
import os

from falconpy import SpotlightVulnerabilities
from rich import print as rprint
from rich.table import Table

client_id = os.environ.get("FALCON_CLIENT_ID", "")
client_secret = os.environ.get("FALCON_CLIENT_SECRET", "")

vulns = SpotlightVulnerabilities(client_id=client_id, client_secret=client_secret)


def show(title, fields):
    table = Table(title=title)
    table.add_column("field", style="cyan")
    table.add_column("value")
    for k, v in fields.items():
        table.add_row(str(k), str(v))
    rprint(table)

## Explore the endpoint

You are using `SpotlightVulnerabilities` again, the same service class from notebook 02. The
method is `query_vulnerabilities_combined` with a `filter` and a `facet`.

The cell below shows how adding filter terms narrows the result count. The fleet has tens of
thousands of open findings. Filtering to just the CISA KEV list brings that down to a small
number.

In [ ]:
open_count = vulns.query_vulnerabilities_combined(
    limit=1, filter="status:'open'", facet=["cve"]
)
print("all open findings in the fleet:", open_count["body"]["meta"]["pagination"]["total"])

kev_count = vulns.query_vulnerabilities_combined(
    limit=1, filter="status:'open'+cve.is_cisa_kev:true", facet=["cve"]
)
print("narrowed to CISA KEV only:", kev_count["body"]["meta"]["pagination"]["total"])

In [ ]:
# TODO: paste the cve_id from notebook 02
cve_id = None

show("Checking exposure to", {"cve": cve_id})

## What this CVE is

Before checking the blast radius, read the detail on this CVE. The cell below queries one open
finding for it and prints the severity, score, exploit status, and adversaries.

This is the same `query_vulnerabilities_combined` call you used in notebook 02, just filtered to
`cve.id` instead of `aid`.

In [ ]:
if not cve_id:
    print("paste the cve_id in the cell above first")
else:
    rec = vulns.query_vulnerabilities_combined(
        limit=1, filter=f"cve.id:'{cve_id}'+status:'open'", facet=["cve"]
    )["body"]["resources"][0]["cve"]

    show("CVE detail", {
        "id": rec["id"],
        "severity": rec["severity"],
        "base_score": rec["base_score"],
        "exploit_status": rec["exploit_status"],
        "actors": ", ".join(rec.get("actors") or []),
    })

## Which other hosts have it

Same call, but now you want to see all hosts with this CVE open, not just one record.

Filter on `cve.id` and `status:'open'`. Add the `host_info` facet so the results include the
hostname. The count in `meta.pagination.total` is your blast radius.

In [ ]:
# TODO: query open findings of this CVE across the whole fleet
# hint: vulns.query_vulnerabilities_combined(filter=f"cve.id:'{cve_id}'+...", facet=[...])
exposure = None  # replace this line with your query

if not exposure or exposure["status_code"] != 200:
    print("query failed or not filled in yet")
else:
    exposed_host_count = exposure["body"]["meta"]["pagination"]["total"]

    table = Table(title=f"Fleet-wide exposure to {cve_id} ({exposed_host_count} open findings)")
    table.add_column("host", style="cyan")
    for vuln in exposure["body"]["resources"]:
        table.add_row(vuln["host_info"]["hostname"])
    rprint(table)

## Fallback

Run a cell here only if the query above returned nothing (tenant or seed issue). Skip it if you
already have a real value.

In [ ]:
import json

try:
    with open("data/sample_responses/exposure_query.json") as handle:
        response = {"status_code": 200, "body": json.load(handle)}
except FileNotFoundError:
    print("sample file not found")
    response = None

if response:
    exposed_host_count = response["body"]["meta"]["pagination"]["total"]

    table = Table(title=f"Fallback exposure ({exposed_host_count} open findings)")
    table.add_column("host", style="cyan")
    for vuln in response["body"]["resources"]:
        table.add_row(vuln["host_info"]["hostname"])
    rprint(table)

## Carry to 04

Every one of those hosts can be hit the same way this one was. Take `cve_id` and
`exposed_host_count` into notebook 04 to get the fix.